GEETA 4B — C5 · Independent Python verifier.

Zero-tolerance correctness guarantee (brief §2.4): every arithmetic step the
model emits is recomputed by an INDEPENDENT Python engine (sympy + fractions).
The model proposes; Python disposes.

Contract
--------
The model must emit structured JSON steps::

    {"steps": [
        {"step": 1, "expression": "9 + 0",  "result": 9,   "explanation": "add the ones"},
        {"step": 2, "expression": "20 + 20","result": 40,  "explanation": "add the tens"},
        {"step": 3, "expression": "29 + 20","result": 49,  "explanation": "put the parts together"}
    ], "final_answer": 49}

* ``expression`` is pure arithmetic (numbers, + - * / ( ), %, a/b fractions,
  decimal numbers). Anything else (names, calls, comparisons) is rejected —
  a rejected expression is a FAIL verdict (the step is unprovable).
* ``result`` must equal the recomputed value (float tolerance 1e-6 relative;
  Fractions compared exactly).
* ``final_answer`` must match the problem's expected answer (via
  :func:`tools.leak_guard.parse_answer_value` normalisation) — this is what
  makes "0 wrong answers delivered" checkable end-to-end.

Every verdict is logged (JSONL) to feed the dev report + scorecard.
Mismatch -> caller bounds regeneration (default 3 retries, config) ->
fallback "let's check this one together" path.

## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import json
import math
import re
import time
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path
from typing import Any, Optional

import sympy

In [ ]:

# ---------------------------------------------------------------------------
# Safe expression evaluation
# ---------------------------------------------------------------------------

# Whitelist: allow digits, operators, parens, spaces, percent.
_EXPR_STRICT = re.compile(r"^[0-9+\-*/().%\s,]*$")

In [ ]:
# a/b fraction tokens (e.g. "1/2") and bare decimals.
_NUM = re.compile(r"(?<![\w.])(\d+(?:\.\d+)?)(?:\s*/\s*(\d+(?:\.\d+)?))?(?![\w.])")

In [ ]:
_PERCENT = re.compile(r"(\d+(?:\.\d+)?)\s*%")

In [ ]:

# Functions the model MAY use — all map to exact sympy equivalents.
_ALLOWED_FUNCS = {
    "abs": lambda x: abs(x),
    "sqrt": lambda x: sympy.sqrt(x),
    "floor": lambda x: sympy.floor(x),
    "ceil": lambda x: sympy.ceil(x),
    "max": max,
    "min": min,
}

In [ ]:


@dataclass
class StepVerdict:
    step: int
    expression: str
    model_result: Any
    recomputed: Any
    ok: bool
    reason: str

    def to_dict(self) -> dict:
        return {
            "step": self.step,
            "expression": self.expression,
            "model_result": self.model_result,
            "recomputed": _plain(self.recomputed),
            "ok": self.ok,
            "reason": self.reason,
        }

In [ ]:


EvalResult = tuple  # ("num" | "time", float)  — time = minutes of day

In [ ]:

_TIME_RE = re.compile(r"^(\d{1,2}):(\d{2})$")  # colon required (1100 stays a number)

In [ ]:


def _plain(v: Any) -> Any:
    """sympy/Fraction/EvalResult -> json-plain."""
    if v is None or isinstance(v, (bool, str, dict, list)):
        return v
    if isinstance(v, tuple) and len(v) == 2 and v[0] in ("num", "time"):
        kind, f = v
        if kind == "time":
            f = int(f)
            return f"{f // 60}:{f % 60:02d}"
        f = float(f)
        return round(f, 10) if not f.is_integer() else int(f)
    if isinstance(v, Fraction):
        return float(v)
    if isinstance(v, sympy.Basic):
        f = float(v)
        return round(f, 10) if not f.is_integer() else int(f)
    # R1 (session 12): TypeError crashed the M5 real run at problem ~50/249 -
    # a malformed model output put a dict as final_answer (e.g. it echoed
    # part of the steps schema instead of a scalar), and float(dict) raised.
    # This function is JSON-serialisation-only (comparisons use str()/
    # _to_exact() elsewhere and already handle unparseable values safely by
    # returning False/None) - so on any other odd type, degrade to str(v)
    # instead of crashing the whole batch on one bad generation.
    try:
        f = float(v)
    except (TypeError, ValueError):
        return str(v)
    return round(f, 10) if abs(f) < 1e15 or not f.is_integer() else int(f)

In [ ]:


def _eval_sympy(expr: str) -> Optional[Any]:
    """Evaluate a pure-arithmetic expression with sympy. None if unsafe."""
    if not expr or not isinstance(expr, str):
        return None
    s = re.sub(r"\s+", " ", expr).strip()
    s = s.replace(",", "")
    s = re.sub(r"(\d)\s*%\s*of\s*", r"\1/100 * ", s, flags=re.I)  # 10% of 40 -> 10/100 * 40
    s = s.replace(" of ", " * ")
    s = s.replace("%", "/100")
    if not _EXPR_STRICT.fullmatch(s):
        return None
    try:
        v = sympy.sympify(s, locals={name: f for name, f in _ALLOWED_FUNCS.items()},
                          rational=True)
    except (sympy.SympifyError, TypeError, ValueError):
        return None
    if v.free_symbols:
        return None
    if v.has(sympy.Eq, sympy.Ne, sympy.Lt, sympy.Gt, sympy.Le, sympy.Ge):
        return None
    return v

In [ ]:


def eval_expression(expr: str) -> Optional[EvalResult]:
    """Recompute a step expression. Returns ("num", float) for arithmetic,
    ("time", minutes-of-day) for a single time token, or None if the
    expression is not pure arithmetic / un-evaluable (== unprovable step).
    """
    t = (expr or "").strip()
    m = _TIME_RE.fullmatch(t)
    if m:
        h, mm = int(m.group(1)), int(m.group(2))
        if mm < 60 and h < 24:
            return ("time", h * 60 + mm)
    v = _eval_sympy(expr)
    if v is None:
        return None
    try:
        return ("num", float(v))
    except (TypeError, ValueError, ZeroDivisionError):
        return None

In [ ]:


def _to_exact(v: Any) -> Optional[Fraction]:
    """Best exact value for a model-stated result (int/float/fraction-string)."""
    if isinstance(v, bool):
        return None
    if isinstance(v, int):
        return Fraction(v)
    if isinstance(v, float):
        return Fraction(v).limit_denominator(10**9)
    if isinstance(v, str):
        s = v.strip().replace(",", "").replace("$", "").replace("%", "")
        if "/" in s:
            num, den = s.split("/", 1)
            try:
                return Fraction(num) / Fraction(den)
            except (ValueError, ZeroDivisionError):
                return None
        try:
            return Fraction(s).limit_denominator(10**9)
        except (ValueError, ZeroDivisionError):
            return None
    return None

In [ ]:


def compare_results(model_result: Any, recomputed: Optional[EvalResult]) -> bool:
    """True if model_result equals recomputed (exact for rationals, 1e-6
    relative otherwise; time values compared as minutes-of-day)."""
    if recomputed is None:
        return False
    kind, rec = recomputed
    if kind == "time":
        if isinstance(model_result, str) and _TIME_RE.fullmatch(model_result.strip()):
            h, mm = _TIME_RE.fullmatch(model_result.strip()).groups()
            return int(h) * 60 + int(mm) == rec
        return False
    exact = _to_exact(model_result)
    if exact is None:
        return False
    if exact.denominator == 1 and abs(float(exact) - rec) < 1e-9:
        return True
    return abs(float(exact) - rec) <= 1e-6 * max(1.0, abs(rec))

In [ ]:


# ---------------------------------------------------------------------------
# Verifier engine
# ---------------------------------------------------------------------------

@dataclass
class Verdict:
    ok: bool
    final_model_answer: Any = None
    final_expected: Any = None
    steps: list = field(default_factory=list)
    reason: str = ""
    elapsed_ms: float = 0.0

    def to_dict(self) -> dict:
        return {
            "ok": self.ok,
            "final_model_answer": _plain(self.final_model_answer),
            "final_expected": _plain(self.final_expected),
            "steps": [s.to_dict() for s in self.steps],
            "reason": self.reason,
            "elapsed_ms": round(self.elapsed_ms, 2),
        }

In [ ]:


class Verifier:
    """Recomputes every model-emitted step + final answer."""

    def __init__(self, expected_answer: Any = None):
        self.expected_answer = expected_answer

    def verify(self, model_output: str | dict, expected_answer: Any = None) -> Verdict:
        t0 = time.perf_counter()
        expected = expected_answer if expected_answer is not None else self.expected_answer
        steps: list[StepVerdict] = []
        data: dict | None = None

        if isinstance(model_output, dict):
            data = model_output
        else:
            data = extract_structured_output(model_output)
            if data is None:
                ms = (time.perf_counter() - t0) * 1000
                return Verdict(False, reason="no structured JSON found in model output",
                               elapsed_ms=ms)

        bad = []
        for i, st in enumerate(data.get("steps") or [], start=1):
            if not isinstance(st, dict):
                bad.append(StepVerdict(i, "?", None, None, False, "step is not an object"))
                continue
            expr = str(st.get("expression", "")).strip()
            result = st.get("result")
            recomputed = eval_expression(expr)
            if recomputed is None:
                # Session 13 fix: a step whose expression is NOT parseable
                # arithmetic (e.g. "Count the sides of each polygon.", or a
                # null/narrative step for a word/shape/time problem) makes no
                # falsifiable numeric claim - it is neither verified-correct
                # nor verified-wrong. Previously this unconditionally failed
                # the WHOLE plan even when the final_answer was exactly
                # right, which is why every non-arithmetic-kind problem
                # (shapes, word-form numbers, some time/money problems) was
                # scoring ~0% despite the model getting them right. Zero-
                # tolerance is unaffected: a step WITH a parseable expression
                # is still checked exactly (see the `else` branch below), and
                # the final answer is still independently verified against
                # the ground-truth `expected` a few lines down.
                ok, reason = True, f"not an arithmetic claim (skipped): {expr!r}"
            else:
                ok = compare_results(result, recomputed)
                reason = "ok" if ok else (
                    f"model said {result!r}, recomputed {recomputed}")
            steps.append(StepVerdict(int(st.get("step", i)), expr, result,
                                      recomputed, ok, reason))
            if not ok:
                bad.append(steps[-1])

        # final answer check (exact via answer normalisation)
        final_ok = True
        final_reason = "no final_answer field (step-level check only)"
        final_model = data.get("final_answer",
                               steps[-1].model_result if steps else None)
        if expected is not None:
            from geeta.leak_guard import parse_answer_value
            k1, v1 = parse_answer_value(str(final_model))
            k2, v2 = parse_answer_value(str(expected))
            final_ok = _values_equal(k1, v1, k2, v2)
            final_reason = ("ok" if final_ok else
                            f"model final {_plain(final_model)!r} != expected {_plain(expected)!r}")

        ok = not bad and final_ok
        ms = (time.perf_counter() - t0) * 1000
        return Verdict(ok, final_model, expected, steps,
                        final_reason if not final_ok else (bad[0].reason if bad else "ok"),
                        ms)

In [ ]:


def _values_equal(k1, v1, k2, v2) -> bool:
    if v1 is None or v2 is None:
        return False
    if k1 == "num" and k2 == "num":
        return abs(float(v1) - float(v2)) <= 1e-6 * max(1.0, abs(float(v1)))
    if k1 == "time" and k2 == "time":
        return int(v1) == int(v2)
    if k1 == "word" and k2 == "word":
        return str(v1).lower() == str(v2).lower()
    # cross-kind: try numeric compare
    try:
        return abs(float(v1) - float(v2)) <= 1e-6 * max(1.0, abs(float(v1)))
    except (TypeError, ValueError):
        return False

In [ ]:


# ---------------------------------------------------------------------------
# Extracting structured JSON from raw model text
# ---------------------------------------------------------------------------

_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.S)

In [ ]:


def extract_structured_output(text: str) -> Optional[dict]:
    """Find the JSON object with "steps" in model text (fenced or bare)."""
    if not text:
        return None
    candidates = []
    for m in _FENCE.finditer(text):
        candidates.append(m.group(1))
    for m in re.finditer(r"\{.*?\}\s*$", text, re.S):
        candidates.append(m.group(0))
    candidates.append(text)
    # longest-brace spans
    for m in re.finditer(r"\{", text):
        depth, i = 0, m.start()
        while i < len(text):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    candidates.append(text[m.start():i + 1])
                    break
            i += 1
    for c in candidates:
        c = c.strip()
        try:
            obj = json.loads(c)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            if "steps" in obj:
                return obj
            if "expression" in obj and "result" in obj:
                return {"steps": [obj], "final_answer": obj.get("result")}
    return None

In [ ]:


# ---------------------------------------------------------------------------
# Orchestrated verify loop (bounded retries + fallback) — brief C5
# ---------------------------------------------------------------------------

class VerifiedPlan:
    """Runs the model->verify loop with bounded retries.

    ``generator(prompt) -> str`` is the model call (client or mock).
    Returns (plan_dict | None, verdict_log, fallback_used).
    """

    def __init__(self, verifier: Verifier, max_retries: int = 3,
                 fallback_message: str = "Let's check this one together — "
                                         "can you show me how you'd start it?"):
        self.verifier = verifier
        self.max_retries = max_retries
        self.fallback_message = fallback_message

    def run(self, generator, prompt: str) -> tuple[Optional[dict], list[dict], bool]:
        log: list[dict] = []
        last: dict | None = None
        for attempt in range(self.max_retries + 1):
            raw = generator(prompt)
            v = self.verifier.verify(raw)
            entry = v.to_dict()
            entry["attempt"] = attempt + 1
            entry["prompt_preview"] = prompt[:120]
            log.append(entry)
            if v.ok:
                data = extract_structured_output(raw) if isinstance(raw, str) else raw
                return data, log, False
            last = extract_structured_output(raw) if isinstance(raw, str) else raw
            prompt = self._retry_prompt(raw, v, attempt)
        return last, log, True

    def _retry_prompt(self, raw: str, v: Verdict, attempt: int) -> str:
        fails = [s for s in v.steps if not s.ok]
        detail = "; ".join(f"step {s.step} '{s.expression}': you said {s.model_result!r}, "
                           f"recompute gives {s.recomputed!r}" for s in fails[:3]) or v.reason
        return (f"Your previous working did not check out: {detail}. "
                "Recompute carefully and emit ONLY the JSON steps object again. "
                "Do not show the final answer to the child yet.")

In [ ]:


class VerdictLog:
    """Append-only JSONL verdict log (feeds dev report + scorecard)."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.touch(exist_ok=True)

    def record(self, verdict: Verdict, meta: dict | None = None) -> None:
        row = {"ts": time.strftime("%Y-%m-%dT%H:%M:%S%z"), **(meta or {})}
        row.update(verdict.to_dict())
        with self.path.open("a") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    def read(self) -> list[dict]:
        if not self.path.exists():
            return []
        return [json.loads(l) for l in self.path.read_text().splitlines() if l.strip()]

In [ ]:


# ---------------------------------------------------------------------------
# Deterministic pre-solver (session 13) - "the model proposes; Python disposes"
# extended: for question shapes Python can solve EXACTLY, Python proposes too.
# Motivation: money questions were ~11% because the LoRA emits bracket-list
# garbage instead of JSON for that shape, so no verified plan existed at all -
# which also disabled leak-guard + solved-detection for those sessions.
# Strictly conservative: returns None unless the parse is unambiguous.
# ---------------------------------------------------------------------------

_MONEY_TOK = re.compile(r"\$\s*(\d+(?:\.\d{1,2})?)|(\d+(?:\.\d+)?)\s*c\b", re.I)
_BARE_EXPR = re.compile(r"^[\s\d+\-*/x×÷().]+$", re.I)


def _money_to_cents(text: str) -> Optional[list]:
    """['$1', '50c'] -> [100, 50]. None unless every money token parses."""
    toks = _MONEY_TOK.findall(text)
    if len(toks) < 2:
        return None
    cents = []
    for dollars, c in toks:
        if dollars:
            cents.append(int(round(float(dollars) * 100)))
        elif c:
            cents.append(int(round(float(c))))
        else:
            return None
    return cents


def _fmt_money(cents: int) -> str:
    return f"${cents // 100}.{cents % 100:02d}"


def try_deterministic_solve(question_text: str) -> Optional[dict]:
    """Return a plan dict {steps, final_answer} for question shapes Python can
    solve exactly, else None. Never guesses: an unclear question returns None
    and the normal model->verify path runs instead."""
    if not question_text or not isinstance(question_text, str):
        return None
    q = question_text.strip()
    low = q.lower()

    # --- 1. money totals: "What is the total? 50c + 5c + $1" -----------------
    if ("$" in q or re.search(r"\dc\b", low)) and any(
            w in low for w in ("total", "altogether", "how much", "sum", "add")):
        # decline anything that isn't a plain token sum: quantity words
        # ("two 20c coins"), multipliers, or change/spend framing all mean the
        # arithmetic is NOT just "add these tokens up".
        _QUANTITY = ("one", "two", "three", "four", "five", "six", "seven",
                     "eight", "nine", "ten", "pair", "dozen", "each", "every",
                     "per", "times", "double", "twice")
        _CONTEXT = ("change", "left", "spend", "spent", "share", "give", "gave",
                    "buy", "bought", "cost", "more than", "less than")
        tail = _MONEY_TOK.sub(" ", q)          # text with the money tokens removed
        tail_words = set(re.findall(r"[a-z]+", tail.lower()))
        joiners = {"what", "is", "the", "total", "altogether", "how", "much",
                   "sum", "add", "and", "plus", "of", "in", "a", "it", "you",
                   "have", "has", "are", "there", "money", "together"}
        if (not (tail_words & set(_QUANTITY)) and not any(w in low for w in _CONTEXT)
                and not (tail_words - joiners)):
            cents = _money_to_cents(q)
            if cents and len(cents) >= 2:
                total = sum(cents)
                expr = " + ".join(str(c) for c in cents)
                return {"steps": [{"step": 1, "expression": expr, "result": total,
                                   "explanation": "convert every coin/note to cents and add"}],
                        "final_answer": _fmt_money(total)}

    # --- 1b. change from a note: "buy X for $A and pay with a $B note" ------
    if "change" in low:
        m = re.search(r"for\s*\$\s*(\d+(?:\.\d{1,2})?)", q, re.I)
        n = re.search(r"with\s+(?:a\s+)?\$\s*(\d+(?:\.\d{1,2})?)", q, re.I)
        if m and n and len(_MONEY_TOK.findall(q)) == 2:
            paid = int(round(float(n.group(1)) * 100))
            cost = int(round(float(m.group(1)) * 100))
            if paid >= cost:
                return {"steps": [{"step": 1, "expression": f"{paid} - {cost}", "result": paid - cost,
                                   "explanation": "in cents, take the price away from what you paid"}],
                        "final_answer": _fmt_money(paid - cost)}

    # --- 2. a bare arithmetic expression: "34 + 27 = ___", "9 x 9" -----------
    stripped = re.sub(r"=\s*[_\s?]*$", "", q).strip()
    stripped = stripped.rstrip("=?").strip()
    if _BARE_EXPR.fullmatch(stripped) and re.search(r"\d", stripped):
        norm = stripped.replace("x", "*").replace("×", "*").replace("÷", "/")
        val = eval_expression(norm)
        if val and val[0] == "num":
            n = val[1]
            out = int(n) if abs(n - round(n)) < 1e-9 else round(n, 4)
            return {"steps": [{"step": 1, "expression": norm, "result": out,
                               "explanation": "work the expression left to right"}],
                    "final_answer": out}
    return None
